<a href="https://colab.research.google.com/github/masumeaslanii/ML2026/blob/main/udlbook/Ch11_3_Residual_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this if you're in a Colab to install MNIST 1D repository
!pip install git+https://github.com/greydanus/mnist1d

  Cloning https://github.com/greydanus/mnist1d to /tmp/pip-req-build-azp8_eo7
  Running command git clone --filter=blob:none --quiet https://github.com/greydanus/mnist1d /tmp/pip-req-build-azp8_eo7
  Resolved https://github.com/greydanus/mnist1d to commit e2f536fa86442c13473b433d579e9852ec4d1c1d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [3]:
args = mnist1d.data.get_dataset_args()
data = mnist1d.data.get_dataset(args, path='./mnist1d_data.pkl', download=False, regenerate=False)

print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Length of each example: {}".format(data['x'].shape[-1]))

Successfully loaded data from ./mnist1d_data.pkl
Examples in training set: 4000
Examples in test set: 1000
Length of each example: 40


In [4]:
train_data_x = data['x'].transpose()
train_data_y = data['y']
val_data_x = data['x_test'].transpose()
val_data_y = data['y_test']

print("Train data: %d examples (columns), each of which has %d dimensions (rows)"%((train_data_x.shape[1],train_data_x.shape[0])))
print("Validation data: %d examples (columns), each of which has %d dimensions (rows)"%((val_data_x.shape[1],val_data_x.shape[0])))

Train data: 4000 examples (columns), each of which has 40 dimensions (rows)
Validation data: 1000 examples (columns), each of which has 40 dimensions (rows)


## The `print_variance` Function

The purpose of this function is to calculate and print the **average variance of neuron outputs across a batch of samples**.


**1. Convert the Tensor to a NumPy array**

- `detach()` disconnects the tensor from the computational graph, so gradients are no longer tracked for this operation.
- `numpy()` converts the PyTorch tensor into a NumPy array, allowing us to use NumPy's statistical functions.

**2. Calculate the variance of each neuron**

Setting `axis=0` means that the variance is calculated separately for each column. In other words, we measure how much each neuron's output changes across different samples in the batch.

**3. Calculate the average variance**

After calculating the variance of each neuron, `np.mean()` computes the average of all these variance values.

Mathematically:
$
\text{Mean Variance} = \frac{1}{N}\sum_{i=1}^{N} \operatorname{Var}(a_i)
$

where \(N\) is the number of neurons and \(a_i\) represents the outputs of neuron \(i\) across the batch.

**4. Print the result**

This prints the name of the layer or data, along with its average neuron variance.

> **Interpretation:**
>
> - **High variance:** Neuron outputs change considerably across different input samples.
> - **Low variance:** Neuron outputs remain relatively similar across different samples.
> - **Zero variance:** A neuron's output is identical for all samples in the batch.

**Overall:** This function measures how much neuron activations vary across a batch of samples and then calculates the average of these variations across all neurons. This can help us investigate how information and activations behave across different layers of a neural network.

In [5]:
def print_variance(name, data):

  np_data = data.detach().numpy()

  neuron_variance = np.mean(np.var(np_data, axis=0))

  print("%s variance=%f"%(name,neuron_variance))

In [6]:
def weights_init(layer_in):
  if isinstance(layer_in, nn.Linear):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

## The `run_one_step_of_model` Function

The purpose of this function is to perform **one training step** of a neural network. It takes one batch of training data, computes the model's predictions and loss, calculates gradients through backpropagation, and updates the model parameters using SGD.

### 1. Define the Loss Function

We use `CrossEntropyLoss` to measure the difference between the model's predictions and the actual labels.

It is commonly used for classification tasks.

### 2. Initialize the Optimizer

Here, we define Stochastic Gradient Descent (SGD) as the optimizer.

- `model.parameters()`: Specifies the trainable parameters (weights and biases) of the model.
- `lr=0.05`: Sets the learning rate, which controls the size of parameter updates.
- `momentum=0.9`: Uses information from previous updates to guide the current updates, helping reduce oscillations and accelerate optimization in relevant directions.

### 3. Create the DataLoader

The DataLoader prepares the training data for batch processing.

- `TensorDataset(x_train, y_train)`: Combines the input samples and their corresponding labels into a dataset.
- `batch_size=200`: Each batch contains 200 samples.
- `shuffle=True`: Randomly changes the order of the samples before creating batches.
- `worker_init_fn`: Defines a function to initialize the random seed for DataLoader workers.

**Note:** In this code, `np.random.seed(1)` is called immediately rather than being passed as a function. Therefore, it does not work as an appropriate worker initialization function. Also, since `num_workers` defaults to zero, worker initialization is not needed in this example.

### 4. Initialize the Model Weights

This applies the `weights_init` function recursively to all modules in the model.

The purpose is to initialize the weights and biases before training begins.

### 5. Retrieve a Batch of Data

The loop iterates over the batches provided by the DataLoader.

Each batch is divided into two parts:

- `x_batch`: The input samples.
- `y_batch`: The corresponding ground-truth labels.

Since `shuffle=True`, the samples are presented in a randomized order.

### 6. Reset the Gradients

Before calculating new gradients, we set the existing gradients to zero.

This is necessary because PyTorch accumulates gradients by default. Without this step, gradients from previous iterations would be added to the newly calculated gradients.

### 7. Forward Pass

The input batch is passed through the neural network to generate predictions.

The variable `pred` contains the model's predicted outputs for all samples in the current batch.

### 8. Calculate the Loss

The model's predictions (`pred`) are compared with the actual labels (`y_batch`) to calculate the loss.

A lower loss generally indicates that the model's predictions are closer to the correct labels.

### 9. Backward Pass

This step performs backpropagation and calculates the gradients of the loss with respect to all trainable model parameters.

Mathematically, for each parameter \(\theta\), we calculate:
$
\frac{\partial Loss}{\partial \theta}
$

These gradients indicate how changes in each parameter affect the loss.

### 10. Update the Model Parameters


The optimizer uses the calculated gradients to update the model's weights and biases.

In this case, SGD adjusts the parameters in an attempt to reduce the loss.

### 11. Break the Loop


This statement stops the loop after processing the first batch.

The purpose is to demonstrate a single optimization step rather than train the model completely.

Normally, we would process all batches and repeat the process over multiple epochs.


> **Important:** This function performs only one optimization step using one batch of 200 samples. It does not train the model for a complete epoch.

In [7]:
def run_one_step_of_model(model, x_train, y_train):

  loss_function = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr = 0.05, momentum=0.9)

  data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=200, shuffle=True, worker_init_fn=np.random.seed(1))

  model.apply(weights_init)


  for i, data in enumerate(data_loader):
    x_batch, y_batch = data

    optimizer.zero_grad()
    pred = model(x_batch)
    loss = loss_function(pred, y_batch)
    loss.backward()
    optimizer.step()

    break

In [8]:
x_train = torch.tensor(train_data_x.transpose().astype('float32'))
y_train = torch.tensor(train_data_y.astype('long'))

## Building a Deeper Residual Neural Network

In this section, we define a deeper Residual Neural Network using PyTorch. Similar to the previous model, this network uses shortcut connections to preserve information and improve gradient propagation. However, instead of two residual blocks, it contains **five consecutive residual blocks**.

### 1. Defining the Residual Network Class

We define a class called `ResidualNetwork` that inherits from `torch.nn.Module`.

This allows us to define the network architecture, manage its layers, and access its learnable parameters.

### 2. Initializing the Network

The `__init__()` method initializes the network when an instance of the class is created.

It accepts three parameters:

- `input_size`: Number of input features.
- `output_size`: Number of output dimensions.
- `hidden_size`: Number of neurons in each hidden layer, with a default value of 100.

The `super()` function calls the constructor of the parent class (`torch.nn.Module`) to ensure that the model is properly initialized.

### 3. Defining the Linear Layers

We define seven fully connected (linear) layers:

| Layer | Input Size | Output Size |
|---|---:|---:|
| `linear1` | 40 | 100 |
| `linear2` | 100 | 100 |
| `linear3` | 100 | 100 |
| `linear4` | 100 | 100 |
| `linear5` | 100 | 100 |
| `linear6` | 100 | 100 |
| `linear7` | 100 | 10 |

The first layer transforms the input into a 100-dimensional hidden representation. The next five layers are used inside the residual blocks, and the final layer maps the hidden representation to 10 output values corresponding to the digit classes (0–9).

Keeping the hidden dimensions equal is important because residual connections require the tensors being added together to have compatible shapes.

### 4. Counting the Network Parameters

The `count_params()` method calculates the total number of learnable parameters in the network.

- `self.parameters()` retrieves all learnable parameters, including weights and biases.
- `p.view(-1)` reshapes each parameter tensor into a one-dimensional tensor.
- `.shape[0]` returns the number of elements in that tensor.
- `sum()` adds the number of elements across all parameter tensors.

Therefore, this method returns the total number of weights and biases in the network.

### 5. Implementing the Forward Pass with Five Residual Blocks

The `forward()` method defines how input data passes through the network.

First, the input is passed through `linear1` to produce an initial hidden representation:
$
f=Linear_1(x)
$

Then, five residual blocks are applied sequentially.

Each residual block follows the same structure:

1. Apply the ReLU activation function to the current hidden representation.
2. Pass the activated representation through a linear layer.
3. Add the result to the original hidden representation using a shortcut connection.

The five residual blocks can be expressed mathematically as follows:

**First Residual Block:**
$
res_1=f+Linear_2(ReLU(f))
$

**Second Residual Block:**
$
res_2=res_1+Linear_3(ReLU(res_1))
$

**Third Residual Block:**
$
res_3=res_2+Linear_4(ReLU(res_2))
$

**Fourth Residual Block:**
$
res_4=res_3+Linear_5(ReLU(res_3))
$

**Fifth Residual Block:**
$
res_5=res_4+Linear_6(ReLU(res_4))
$

Finally, the output of the fifth residual block is passed through `linear7` to produce the final predictions:
$
output=Linear_7(res_5)
$

### 6. Monitoring Variance Across the Network

An important addition to this implementation is the use of the `print_variance()` function throughout the forward pass.

We calculate and print the variance at several points:

- **Input:** Variance of the original input features across the batch.
- **First preactivation:** Variance after the first linear transformation, before applying ReLU.
- **After each residual connection:** Variance of the hidden representations after each residual block.

This allows us to observe how the variance of neuron activations changes as information passes through the network.

In particular, we can investigate whether the variance increases, decreases, or remains relatively stable as more residual blocks are added.

The purpose of this analysis is to understand the behavior of activations in a deeper residual network and examine how residual connections affect the propagation of information through its layers.

> **The main idea:** Residual connections allow each block to learn a transformation that is added to its original input, rather than requiring the block to learn an entirely new representation. This provides shortcut paths for information and gradients, which can help with training deeper neural networks.

In [9]:
class ResidualNetwork(torch.nn.Module):
  def __init__(self, input_size, output_size, hidden_size=100):
    super(ResidualNetwork, self).__init__()
    self.linear1 = nn.Linear(input_size, hidden_size)
    self.linear2 = nn.Linear(hidden_size, hidden_size)
    self.linear3 = nn.Linear(hidden_size, hidden_size)
    self.linear4 = nn.Linear(hidden_size, hidden_size)
    self.linear5 = nn.Linear(hidden_size, hidden_size)
    self.linear6 = nn.Linear(hidden_size, hidden_size)
    self.linear7 = nn.Linear(hidden_size, output_size)

  def count_params(self):
    return sum([p.view(-1).shape[0] for p in self.parameters()])

  def forward(self, x):
    print_variance("Input",x)
    f = self.linear1(x)
    print_variance("First preactivation",f)
    res1 = f+ self.linear2(f.relu())
    print_variance("After first residual connection",res1)
    res2 = res1 + self.linear3(res1.relu())
    print_variance("After second residual connection",res2)
    res3 = res2 + self.linear4(res2.relu())
    print_variance("After third residual connection",res3)
    res4 = res3 + self.linear5(res3.relu())
    print_variance("After fourth residual connection",res4)
    res5 = res4 + self.linear6(res4.relu())
    print_variance("After fifth residual connection",res5)

    return self.linear7(res5)

## Defining the Residual Network and Running One Training Step

First, we define the dimensions of the network:

- `n_input = 40`: The number of input features.
- `n_hidden = 100`: The number of neurons in each hidden layer.
- `n_output = 10`: The number of output classes.

Next, we create an instance of `ResidualNetwork` using these dimensions:
`
model = ResidualNetwork(n_input, n_output, n_hidden)
`

Finally, we pass the model and the training data to `run_one_step_of_model()`.

This function performs **only one training step** using the first batch of 200 training samples. During this step, it initializes the model's weights, performs a forward pass, calculates the loss, computes gradients through backpropagation, and updates the model parameters using SGD.

During the forward pass, the `print_variance()` function monitors the variance of the activations at different points in the network, including the input, the first preactivation, and the outputs after each of the five residual connections.

> **Note:** Since the function contains a `break` statement after processing the first batch, the model is not trained for a complete epoch. The purpose of this experiment is **to observe how the variance changes** as data passes through the residual network.

In [10]:
n_hidden = 100
n_input = 40
n_output = 10
model = ResidualNetwork(n_input, n_output, n_hidden)
run_one_step_of_model(model, x_train, y_train)

Input variance=0.959295
First preactivation variance=2.042205
After first residual connection variance=3.584321
After second residual connection variance=6.067986
After third residual connection variance=9.493089
After fourth residual connection variance=16.754450
After fifth residual connection variance=29.222462


## Applying Batch Normalization to the Residual Network

In this implementation, Batch Normalization is added **before the ReLU activation in the main branch of each residual block**.

Each residual block has two paths:

- **Skip connection:** Passes the original input directly to the addition operation without any modification.
- **Main branch:** Applies Batch Normalization, followed by ReLU and a Linear layer.

For example, the first residual block is defined as:

```python
res1 = f + self.linear2(self.bn1(f).relu())
```

The operations are performed in the following order:

1. `self.bn1(f)`: Applies Batch Normalization to the input `f`.
2. `.relu()`: Applies the ReLU activation function to the normalized values.
3. `self.linear2(...)`: Passes the result through a Linear layer.
4. `f + ...`: Adds the result to the original input through the skip connection.

The same procedure is repeated for all five residual blocks:

| Residual Block | Batch Normalization Input |
|---|---|
| First | `f` |
| Second | `res1` |
| Third | `res2` |
| Fourth | `res3` |
| Fifth | `res4` |

We use `nn.BatchNorm1d(hidden_size)` because the network processes two-dimensional data with the shape `(batch_size, hidden_size)`.

> **Important:** Batch Normalization is applied only to the main branch, not to the skip connection. This allows the original input to bypass the normalization and transformation operations before being added to the main branch.

In [11]:
class ResidualNetworkWithBatchNorm(torch.nn.Module):
  def __init__(self, input_size, output_size, hidden_size=100):
    super(ResidualNetworkWithBatchNorm, self).__init__()

    self.linear1 = nn.Linear(input_size, hidden_size)
    self.linear2 = nn.Linear(hidden_size, hidden_size)
    self.linear3 = nn.Linear(hidden_size, hidden_size)
    self.linear4 = nn.Linear(hidden_size, hidden_size)
    self.linear5 = nn.Linear(hidden_size, hidden_size)
    self.linear6 = nn.Linear(hidden_size, hidden_size)

    self.bn1 = nn.BatchNorm1d(hidden_size)
    self.bn2 = nn.BatchNorm1d(hidden_size)
    self.bn3 = nn.BatchNorm1d(hidden_size)
    self.bn4 = nn.BatchNorm1d(hidden_size)
    self.bn5 = nn.BatchNorm1d(hidden_size)

    self.linear7 = nn.Linear(hidden_size, output_size)

  def count_params(self):
    return sum([p.view(-1).shape[0] for p in self.parameters()])

  def forward(self, x):
    print_variance("Input", x)

    f = self.linear1(x)
    print_variance("First preactivation", f)

    res1 = f + self.linear2(self.bn1(f).relu())
    print_variance("After first residual connection", res1)

    res2 = res1 + self.linear3(self.bn2(res1).relu())
    print_variance("After second residual connection", res2)

    res3 = res2 + self.linear4(self.bn3(res2).relu())
    print_variance("After third residual connection", res3)

    res4 = res3 + self.linear5(self.bn4(res3).relu())
    print_variance("After fourth residual connection", res4)

    res5 = res4 + self.linear6(self.bn5(res4).relu())
    print_variance("After fifth residual connection", res5)

    return self.linear7(res5)


In [12]:
n_hidden = 100
n_input = 40
n_output = 10
model = ResidualNetworkWithBatchNorm(n_input, n_output, n_hidden)
run_one_step_of_model(model, x_train, y_train)

Input variance=0.993989
First preactivation variance=1.923312
After first residual connection variance=2.578809
After second residual connection variance=3.393499
After third residual connection variance=4.146667
After fourth residual connection variance=4.768885
After fifth residual connection variance=5.585710


## Comparing Residual Networks With and Without Batch Normalization

The following table compares the activation variance at different points in the two residual networks.

| Measurement Point | Without BatchNorm | With BatchNorm |
|---|---:|---:|
| Input | 0.991639 | 0.933248 |
| First preactivation | 2.074166 | 1.931443 |
| After first residual connection | 3.422480 | 2.666518 |
| After second residual connection | 5.366496 | 3.211927 |
| After third residual connection | 9.171859 | 3.903073 |
| After fourth residual connection | 14.382878 | 4.614060 |
| After fifth residual connection | 25.832754 | 5.344524 |

**Without Batch Normalization:**

The variance increases significantly as the data passes through the network. It starts at approximately 0.99 and reaches 25.83 after the fifth residual connection.

This suggests that the activations become increasingly spread out across the network. The repeated addition of the transformed input to the original input can contribute to this increase in variance.

**With Batch Normalization:**

The variance still increases across the residual blocks, but the growth is considerably smaller. It starts at approximately 0.93 and reaches only 5.34 after the fifth residual connection.

This happens because Batch Normalization normalizes the input to each residual branch before the ReLU activation and Linear transformation. As a result, the values entering the Linear layers are more controlled, helping to limit the growth of activation variance.

**Conclusion:**
Batch Normalization significantly reduces the increase in activation variance throughout the residual network. In this experiment, the final variance decreases from approximately 25.83 without Batch Normalization to 5.34 with Batch Normalization.

**This suggests that Batch Normalization helps control activation variance and can improve the stability of training deep neural networks.**

> Note: These results are obtained from only one training step. Therefore, they demonstrate the effect of Batch Normalization on activation variance, but do not necessarily prove that the model achieves better accuracy or generalization.